In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
import bs4
#from langchain import hub
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("LOR.txt")

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [4]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [5]:
output = rag_chain.invoke({"input": "Hi?"})
output["answer"]

'Hello! How can I assist you today?'

In [6]:
output = rag_chain.invoke({"input": "How are you"})
output["answer"]

"I don't have feelings, but I'm here and ready to assist you! How can I help you today?"

In [7]:
output = rag_chain.invoke({"input": "I'm very bored today"})
output["answer"]

"I'm sorry to hear that you're feeling bored. Maybe you could read a book, watch a movie, or try a new hobby to pass the time. If you're interested, I can suggest some activities or topics to explore!"

In [8]:
output = rag_chain.invoke({"input": "Who Made the one ring?"})
output["answer"]

'The One Ring was created by the Dark Lord Sauron.'